# Entrenamiento de Huellas-GAN en Colab

Receta **camino B** del proyecto: cDCGAN condicional + DiffAugment + EMA + TTUR + Upsample-Conv en el Generator. Entrenamos en una GPU T4 de Colab Free.

**Qué va a pasar:**

1. Colab te da una máquina virtual temporal con GPU.
2. Clonamos el repo desde GitHub.
3. Subís `kaggle.json` una sola vez (para que bajemos el dataset SOCOFing).
4. Regeneramos el dataset dentro de la VM (~10 min).
5. Entrenamos la GAN (~2.5-3 h en T4 con los 150 epochs default; DiffAug agrega ~10-15% por step).
6. Al final te bajás `generator.pt` (que es el **G_EMA**) y los samples a tu PC.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

> **Estado al 2026-04-26 (camino B):** nueva arquitectura del Generator (Upsample+Conv 3x3 en vez de ConvTranspose2d 4x4 stride=2 — evita checkerboard) + DiffAugment policy `"translation,cutout"` aplicado a real y fake antes del D + EMA decay 0.999 sobre el Generator + TTUR `lr_g=1e-4, lr_d=4e-4`. La celda 5.5 default va a `huellas_gan_camino_b/` para no pisar v1 (Fase 5 shipping) ni v2/v3.
>
> **Importante:** los checkpoints de v1/v2 (Fase 5) NO son compatibles con el camino B porque cambió el `state_dict` del Generator. La celda 6.1 (resume) solo sirve para retomar una corrida del camino B que se cortó.

## 1. Verificar GPU

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Mem total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('OJO: no hay GPU. Entorno de ejecución -> Cambiar tipo de entorno -> GPU.')

## 2. Clonar el repo

El código se trae desde GitHub al disco de la VM.

In [ ]:
%cd /content
!rm -rf /content/huellas
!git clone https://github.com/Arielsecchi/huellas.git /content/huellas
%cd /content/huellas

# Flush Python module cache: si el kernel ya importo src.* de una corrida
# anterior, los modulos quedan cacheados en memoria aunque el clone traiga
# codigo nuevo al disco. Esto los descarga para que el proximo import lea
# del disco recien clonado.
import sys
for _m in list(sys.modules):
    if _m == 'src' or _m.startswith('src.'):
        del sys.modules[_m]

## 3. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 4. Subir `kaggle.json`

Te va a aparecer un botón **"Elegir archivos"**. Subí tu `kaggle.json` desde la PC (es el archivo que bajaste de kaggle.com → Settings → API → Create New Token).

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Subí el archivo kaggle.json'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json instalado OK')

## 5. Descargar SOCOFing + preproceso + etiquetado Vucetich

Las tres fases del pipeline de datos corren en <10 min totales. El resultado (`data/processed/images.npz` y `data/processed/metadata.csv`) queda en el disco local de la VM.

In [ ]:
!python -m src.data.download_socofing

In [ ]:
!python -m src.data.preprocess

In [ ]:
!python -m src.data.label_vucetich --full

## 5.5. (Recomendado) Montar Google Drive

Los archivos en `/content/` se borran cuando la VM se desconecta (por inactividad o cuota agotada). Para NO perder los checkpoints si la sesión se corta antes de terminar, montamos Drive y guardamos ahí.

Si saltás esta celda, el entrenamiento corre igual pero los resultados son volátiles.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# camino B (2026-04-26): Upsample+Conv + DiffAugment + EMA + TTUR.
# Carpeta nueva para no pisar v1/v2 que tienen otra arquitectura.
DRIVE_ROOT = Path('/content/drive/MyDrive/huellas_gan_camino_b')
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'final').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'training_samples').mkdir(parents=True, exist_ok=True)
print(f'[ok] outputs van a: {DRIVE_ROOT}')

## 6. Entrenamiento completo

150 épocas con batch 64 tardan **~2.5-3 h en una T4** con la receta camino B (cBN en G + Upsample+Conv + hinge + SN en D + horizontal flip con swap I↔E + **DiffAugment** + **EMA del Generator** + **TTUR** lr_d=4e-4 / lr_g=1e-4).

Los checkpoints cada 25 épocas y los samples cada 5 épocas se guardan en Drive (si corriste la celda 5.5). Si se corta la sesión antes de terminar, podés continuar desde el último ckpt con la celda 6.1.

> **Importante:** no cierres la pestaña de Colab durante el entrenamiento. Si la VM se reinicia, se pierde todo lo que está en `/content/`, pero los checkpoints y samples guardados en Drive sobreviven.
>
> **Nota:** los samples que vas a ver en Drive son del **G_EMA**, no del G crudo — así que lo que ves es la calidad real de inferencia que va a tener la app.

In [ ]:
from pathlib import Path

from src.training.config import TrainConfig
from src.training.train import train

# usamos Drive si se monto en la celda 5.5; si no, guardamos local
try:
    DRIVE_ROOT
    paths = dict(
        checkpoints_dir=DRIVE_ROOT / 'checkpoints',
        final_model_path=DRIVE_ROOT / 'final' / 'generator.pt',
        samples_dir=DRIVE_ROOT / 'training_samples',
        log_path=DRIVE_ROOT / 'training_samples' / 'train_log.csv',
    )
    print(f'[drive] outputs a {DRIVE_ROOT}')
except NameError:
    paths = {}
    print('[local] Drive no montado; outputs se pierden si la VM se desconecta')

cfg = TrainConfig(
    epochs=150,
    batch_size=64,
    num_workers=2,
    sample_every_epochs=5,
    ckpt_every_epochs=25,
    **paths,
)
train(cfg)

### 6.1. (Alternativa) Resumir desde checkpoint

Si la sesión anterior se cortó antes de terminar las 150 épocas, corré ESTA celda en vez de la 6 para continuar desde el último `ckpt_XXX.pt` que tengas en Drive.

Requiere haber montado Drive (celda 5.5) y que existan checkpoints **del camino B** en `huellas_gan_camino_b/checkpoints/`. Editá la variable `RESUME_EPOCH` abajo para apuntar al último ckpt guardado (se guarda uno cada 25 épocas → 025, 050, 075, 100, 125).

> **Atención:** los checkpoints viejos de Fase 5 (`huellas_gan/`, `huellas_gan_v2/`) NO sirven para resumir acá — el `state_dict` cambió porque el camino B usa Upsample+Conv en vez de ConvTranspose2d en el Generator. Si querés retomar una corrida vieja, necesitás también revertir los commits del camino B.

In [ ]:
from src.training.config import TrainConfig
from src.training.train import train

RESUME_EPOCH = 75  # editar al ultimo ckpt guardado (025, 050, 075, 100, 125)

ckpt_path = DRIVE_ROOT / 'checkpoints' / f'ckpt_{RESUME_EPOCH:03d}.pt'
assert ckpt_path.exists(), f'No encuentro {ckpt_path}. Listá Drive/huellas_gan/checkpoints y ajustá RESUME_EPOCH.'

cfg = TrainConfig(
    epochs=150,
    batch_size=64,
    num_workers=2,
    sample_every_epochs=5,
    ckpt_every_epochs=25,
    checkpoints_dir=DRIVE_ROOT / 'checkpoints',
    final_model_path=DRIVE_ROOT / 'final' / 'generator.pt',
    samples_dir=DRIVE_ROOT / 'training_samples',
    log_path=DRIVE_ROOT / 'training_samples' / 'train_log.csv',
    resume_from=ckpt_path,
)
train(cfg)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

samples = sorted(Path('outputs/training_samples').glob('epoch_*.png'))
print(f'{len(samples)} grillas de samples')
if samples:
    display(Image(filename=str(samples[-1])))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# lee de Drive si se monto, sino de local
try:
    samples_dir = DRIVE_ROOT / 'training_samples'
except NameError:
    samples_dir = Path('outputs/training_samples')

samples = sorted(samples_dir.glob('epoch_*.png'))
print(f'{len(samples)} grillas de samples en {samples_dir}')
if samples:
    display(Image(filename=str(samples[-1])))

import csv
import matplotlib.pyplot as plt
from pathlib import Path

try:
    log_path = DRIVE_ROOT / 'training_samples' / 'train_log.csv'
except NameError:
    log_path = Path('outputs/training_samples/train_log.csv')

with open(log_path) as f:
    rows = list(csv.DictReader(f))
epochs = [int(r['epoch']) for r in rows]
loss_d = [float(r['loss_d']) for r in rows]
loss_g = [float(r['loss_g']) for r in rows]

plt.figure(figsize=(8, 4))
plt.plot(epochs, loss_d, label='D')
plt.plot(epochs, loss_g, label='G')
plt.xlabel('época')
plt.ylabel('loss (media de la época)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import shutil
from google.colab import files

# armamos una carpeta limpia con lo que nos llevamos
shutil.rmtree('/content/huellas_out', ignore_errors=True)
shutil.copytree('models/final', '/content/huellas_out/final')
shutil.copytree('outputs/training_samples', '/content/huellas_out/training_samples')

# zip + download
shutil.make_archive('/content/huellas_out', 'zip', '/content/huellas_out')
files.download('/content/huellas_out.zip')

import shutil
from pathlib import Path
from google.colab import files

# tomamos de Drive si se monto, sino de local
try:
    src_final = DRIVE_ROOT / 'final'
    src_samples = DRIVE_ROOT / 'training_samples'
except NameError:
    src_final = Path('models/final')
    src_samples = Path('outputs/training_samples')

shutil.rmtree('/content/huellas_out', ignore_errors=True)
shutil.copytree(src_final, '/content/huellas_out/final')
shutil.copytree(src_samples, '/content/huellas_out/training_samples')

shutil.make_archive('/content/huellas_out', 'zip', '/content/huellas_out')
files.download('/content/huellas_out.zip')